In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 849.9 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached pybind11-2.11.1-py3-none-any.whl (227 kB)
  Created wheel for fasttext: filename=fasttext-0.9.2-cp310-cp310-linux_x86_64.whl size=4199771 sha256=bfb8793fc42d6ae9f1683eb1e62c9c59130b9e14917e28018a3a61c1f558c7d8
  Stored in directory: /root/.cache/pip/wheels/a5/13/75/f811c84a8ab36eedbaef977a6a58a98990e8e0f1967f98f394
Successfully built fasttext


In [ ]:
import pandas as pd
import numpy as np
import json
import fasttext
import re, os, shutil, json
from os import path
from numpy.linalg import norm
import pickle

# Stop Words Processing - for data cleaning
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
sw = set(stopwords.words('english'))
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
drive.mount('/content/gdrive')

defaultPath = "/content/gdrive/MyDrive/FYP1/"
Dataset_Folder = defaultPath + "Dataset/"
Processed_Dataset_Pickle_Path = Dataset_Folder + "processed_dataset.pkl"
Relevant_Comments_List_Pickle_Path = Dataset_Folder + "relevant_comments_list.pkl"
FastTextModelPath = defaultPath + "FastTextModelFile/cc.en.300-001.bin"


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


Reading Dataset Files and Merging them

In [ ]:
Dataset_Files = os.listdir(Dataset_Folder)

In [ ]:
DFs = []
length = 0

for file in Dataset_Files:
  if file.endswith('.csv'):
    df = pd.read_csv(Dataset_Folder + file, usecols=["author", "like_count", "text"])
    DFs.append(df)
    length = length + 1

In [ ]:
# Merging DataFrames

dataset_df = pd.concat(DFs)
dataset_df.shape

(1044, 3)

In [ ]:
dataset_df.head(5)

,author,like_count,text
0,Densi S,0,I have lost 12kgs in 3months
1,Anjaneyudu,0,Hi Channel admin &amp; all !!<br>I&#39;m serio...
2,Rishabh Yadav,2,10 kgs in 20 days Thanks to you didn&#39;t got...
3,Maryum R,0,Can anyone tell me that if I&#39;ll do keto di...
4,C B N,1,"If you wants to go for keto diet,first have to..."


Dataset Preprocessing

In [ ]:
# Defining Text Preprocing Function

def remove_stop_words(text):
  text_tokens = word_tokenize(text)
  tokens_without_sw = [word for word in text_tokens if not word in sw]
  return ' '.join(map(str, tokens_without_sw))


def preprocess(text):

    text = text.lower()
    text = text.replace("\\n", " ").replace("\n", " ")      # remove \n
    text = remove_stop_words(text)                          # remove stop words
    text = re.sub(r'\b\w{1,2}\b', ' ', text)                # remove words with 1 or 2 characters only
    text = re.sub(r'[^ a-zA-Z0-9]+','', text)                  # remove special characters / remove all except characters and spaces

    if text == "nan" or text == "none":
        text = " "
    text = re.sub(' +', ' ', text)

    return text.strip()

FastText - Features Generation

In [ ]:
fastText_model = fasttext.load_model(FastTextModelPath)

In [ ]:
def fasttext_EmbeddingsGenerator(text):
  text_embeddings = fastText_model.get_word_vector(text)
  return text_embeddings #returns one feature vector.

Sentence Transformer - Features Generation

TF-IDF - Features Generation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline

#print("no. of docs, no. of unique corpus words) : ", tfidf_features_array.shape)
#returns two dimensional feature array against the entire corpus(dataset).
def TFIDF_EmbeddingsGenerator(corpus):
  tfidf_model = TfidfVectorizer(stop_words='english')
  tfidf_model = tfidf_model.fit(corpus) #provide dataset.
  tfidf_features = tfidf_model.transform(corpus) #returns feature vectors.
  tfidf_features_array = tfidf_features.toarray() #converts to array.
  tfidf_features_list = []
  for arr in tfidf_features_array:
    tfidf_features_list.append(arr) #appending feature arrays into a list.
  return (tfidf_features_list, tfidf_model)

#generating feature array against the query.
#This function Takes in dataset's feature array and the query.
def TFIDF_QueryFeatureGeneration(tfidf_model, query):
  tfidf_features = tfidf_model.transform([query])
  tfidf_features_array = tfidf_features.toarray()
  return tfidf_features_array[0]

**Creating feature vectors of our Dataset**

In [ ]:
# Looping over dataset to generate feature vectors
processed_text = []
fasttext_features = []
tfidf_features = []

for index, row in dataset_df.iterrows():

  processed = preprocess(row["text"]) #sending data for cleaning.
  processed_text.append(processed) #appending in the list.

  # FastText Features Generator
  text_features = fasttext_EmbeddingsGenerator(processed) #takes in one comment at a time and returns features.
  fasttext_features.append(text_features)

  #Sentence Transformer Features Generator
  #
  #

# TFIDF Features Generator
(tfidf_features, tfidf_model) = TFIDF_EmbeddingsGenerator(processed_text) #passing the corpus. The func returns the trained model and extracted features.

dataset_df["processed_text"] = processed_text
dataset_df["fasttext_features"] = fasttext_features
dataset_df["tfidf_features"] = tfidf_features

dataset_df.to_pickle(Processed_Dataset_Pickle_Path, compression='infer', protocol=5, storage_options=None) #saving features data into pickle file.

**Reading Processed Dataset**

In [ ]:
dataset_df = pd.read_pickle(Processed_Dataset_Pickle_Path) #reading the file.
dataset_df.head(5)

,author,like_count,text,processed_text,fasttext_features,tfidf_features
0,Densi S,0,I have lost 12kgs in 3months,lost 12kgs 3months,"[-0.00079967454, 0.0032817647, -0.010505701, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,Anjaneyudu,0,Hi Channel admin &amp; all !!<br>I&#39;m serio...,channel admin amp seriously looking someones h...,"[0.00035663048, 0.00026676554, -0.005128587, 0...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Rishabh Yadav,2,10 kgs in 20 days Thanks to you didn&#39;t got...,kgs days thanks got overweight air force medic...,"[-0.0036859573, -0.0075673764, 0.014021413, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,Maryum R,0,Can anyone tell me that if I&#39;ll do keto di...,anyone tell keto diet days follow days diet ga...,"[-0.003192219, -0.0044949376, -0.011215435, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,C B N,1,"If you wants to go for keto diet,first have to...",wants keto diet first calculate calaries per w...,"[-0.010290721, -0.0021225929, -0.006684994, 0....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [ ]:
print("fasttext", type(dataset_df["fasttext_features"]))
print("tfidf   ", type(dataset_df["tfidf_features"]))

fasttext <class 'pandas.core.series.Series'>
tfidf    <class 'pandas.core.series.Series'>


**Defining function to extract relevant comments w.r.t query**

In [ ]:
def cosine_similarity(A, B):

  # compute cosine similarity to find the similarity between two vectors.Higher the value, higher the similarity.
  cosine = np.dot(A,B)/(norm(A)*norm(B))
  return cosine


def get_relevant_comments_indices(query, comments, top_relevant_comments):
  #finding out relevant comments by returning indexes.
#comments: list of feature vectors.
  cosine_similarities = {}
  index = 0

  for comment in comments:
    cosine = cosine_similarity(query, comment) #calculating cosine similarity between query and the feature vector in feature vector list.
    cosine_similarities[cosine] = index #making a dictionary - key-value pair, cosine is the key and index is value.
    index = index + 1

  # Sort by key
  sorted_cosine_similarities = dict(sorted(cosine_similarities.items(), reverse = True)) #returning a dict of sorted keys(highest cosine value).
  print(sorted_cosine_similarities)
  # create a list of of relevant comments indices
  relevant_indices = []
  for i in range(0, top_relevant_comments): #top_relevant_comments= count.
    key, value = list(sorted_cosine_similarities.items())[i] #accessing top values to get the index of the comment.
    relevant_indices.append(value)

  return relevant_indices

**Ask user query and find relevant comments**

In [ ]:
query = input()

# query text preprocessing
processed_query = preprocess(query)

# FastText Features Generator
query_FastText_features = fasttext_EmbeddingsGenerator(processed_query)
# TFIDF Features Generator
query_TFIDF_features = TFIDF_QueryFeatureGeneration(tfidf_model, processed_query)


indices = get_relevant_comments_indices(query_TFIDF_features, dataset_df["tfidf_features"].tolist(), 10) #giving feature vectors of query and the dataset.

print(type(query_TFIDF_features), type(dataset_df["tfidf_features"].tolist()))
print(type(query_TFIDF_features[0]), type(dataset_df["tfidf_features"].tolist()[0]))

# I want to adopt keto diet

I want to adopt keto diet
{0.27321206198185893: 33, 0.2192733274438061: 383, 0.20367982207500882: 1017, 0.19954871306342553: 13, 0.1941955560616456: 253, 0.19410480839433053: 12, 0.1861527462803094: 278, 0.1817621447204832: 884, 0.17958500500251673: 375, 0.16126979783738482: 102, 0.1595562029291585: 58, 0.15756444614822301: 323, 0.1553604945904558: 434, 0.15044884135976452: 211, 0.14267379135307193: 877, 0.1419273920318557: 3, 0.1402095314851371: 373, 0.13976436117117322: 9, 0.13704633125129123: 540, 0.13584685026742654: 56, 0.1355817402499555: 929, 0.1342397793080043: 782, 0.13160158629223642: 691, 0.12968635336269893: 52, 0.1296226322335674: 196, 0.12884139111316542: 758, 0.12699480218447254: 10, 0.12635282167215167: 32, 0.1263226313892047: 374, 0.12499999584725402: 880, 0.12422727971575662: 344, 0.12039887203743821: 706, 0.11721728137250036: 255, 0.11559971445252187: 27, 0.11445860347471726: 238, 0.11343064188261469: 832, 0.11274781653582971: 1027, 0.11263339241446707: 46, 0.1108871

In [ ]:
indices

[33, 383, 1017, 13, 253, 12, 278, 884, 375, 102]

In [ ]:
relevancy_count = 1
relevant_comments_list = []
Comments_str = ""

print("Most Relevant Comments w.r.t Query\n")
for index in indices:
  comment = dataset_df.iloc[index]["text"]
  print(str(relevancy_count) + " relevant comment: " + comment)
  relevancy_count = relevancy_count + 1
  relevant_comments_list.append(comment)
  Comments_str = Comments_str + comment + ". "


with open(Relevant_Comments_List_Pickle_Path, 'wb') as f:
  pickle.dump(relevant_comments_list, f)

Most Relevant Comments w.r.t Query

1 relevant comment: But I don&#39;t want to lose weight
2 relevant comment: after keto diet <br>what should I do for not gain weight
3 relevant comment: What can I do to gain weight? I don’t want to eat garbage but I also want to gain weight
4 relevant comment: Just wondering after the keto out, I&#39;ve learnt that I&#39;d better do the 7 day plan to come back to my normal diet in this video. So can I eat whatever I want to eat and exercise to keep the weight after the 7 day plan that you suggested? Would that maintain my weight? I really can&#39;t follow keto diet forever.. I&#39;d rather eat something I want and exercise to keep my weight.. is that a bad idea? I am on keto diet and this is my 2nd week and it is going pretty good (lost 3.5kg) but I just found from other Youtuber (I should have found that youtuber earlier so this would not have happened), that if I once start keto, it should be my lifestyle otherwise not to try. I am confused whethe

In [ ]:
print(relevant_comments_list)

['But I don&#39;t want to lose weight', 'after keto diet <br>what should I do for not gain weight', 'What can I do to gain weight? I don’t want to eat garbage but I also want to gain weight', 'Just wondering after the keto out, I&#39;ve learnt that I&#39;d better do the 7 day plan to come back to my normal diet in this video. So can I eat whatever I want to eat and exercise to keep the weight after the 7 day plan that you suggested? Would that maintain my weight? I really can&#39;t follow keto diet forever.. I&#39;d rather eat something I want and exercise to keep my weight.. is that a bad idea? I am on keto diet and this is my 2nd week and it is going pretty good (lost 3.5kg) but I just found from other Youtuber (I should have found that youtuber earlier so this would not have happened), that if I once start keto, it should be my lifestyle otherwise not to try. I am confused whether to keep on doing keto or not... it is good to lose weight but somehow hard to do it forever.. Could som

**Text Sumarization**

In [ ]:
!pip install evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.3 MB/s eta 0:00:00


In [ ]:
import nltk
import numpy as np
import pickle
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

defaultPath = "/content/gdrive/MyDrive/FYP1/"
Dataset_Folder = defaultPath + "Dataset/"
Processed_Dataset_Pickle_Path = Dataset_Folder + "processed_dataset.pkl"
Relevant_Comments_List_Pickle_Path = Dataset_Folder + "relevant_comments_list.pkl"
FastTextModelPath = defaultPath + "FastTextModelFile/cc.en.300-001.bin"


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


**Read relevant_comments_list**

In [ ]:
relevant_comments_list = []

with open(Relevant_Comments_List_Pickle_Path, 'rb') as f:
  relevant_comments_list = pickle.load(f)

#relevant_comments_str = relevant_comments_list[0]
relevant_comments_str = ". ".join(relevant_comments_list) #merging all comments into a single string.


In [ ]:
#model_name='facebook/bart-large-cnn'
#model_name = "./results/checkpoint-22500"
#model_name='google/flan-t5-base'
#model_name='SyedShaheer/my_awesome_billsum_model'
model_name = "SyedShaheer/my_awesome_billsum_model2"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

tokenizer_config.json:   0%|          | 0.00/344 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

In [ ]:
dialogue = relevant_comments_str

prompt = f"""
Comment:

{dialogue}

Generate a summary?
"""

#generation_config = GenerationConfig(max_new_tokens=10, do_sample=True, temperature=0.5)

inputs = tokenizer(prompt, return_tensors='pt')


output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=100,
        #generation_config=generation_config,
        )[0],
    skip_special_tokens=True
)

print("-----\n")
print(f'INPUT PROMPT:\n{prompt}')

print("-----\n")
print(f'MODEL GENERATION - \n{output}\n')

-----

INPUT PROMPT:

Comment:

But I don&#39;t want to lose weight. after keto diet <br>what should I do for not gain weight. What can I do to gain weight? I don’t want to eat garbage but I also want to gain weight. Just wondering after the keto out, I&#39;ve learnt that I&#39;d better do the 7 day plan to come back to my normal diet in this video. So can I eat whatever I want to eat and exercise to keep the weight after the 7 day plan that you suggested? Would that maintain my weight? I really can&#39;t follow keto diet forever.. I&#39;d rather eat something I want and exercise to keep my weight.. is that a bad idea? I am on keto diet and this is my 2nd week and it is going pretty good (lost 3.5kg) but I just found from other Youtuber (I should have found that youtuber earlier so this would not have happened), that if I once start keto, it should be my lifestyle otherwise not to try. I am confused whether to keep on doing keto or not... it is good to lose weight but somehow hard to d